<div style="background:#0f3460;padding:40px;border-radius:12px;text-align:center;">

<h1 style="color:#e94560;font-size:28px;font-weight:bold;margin:0 0 12px 0;">
  Taller modelación # 1
</h1>

**Integrantes:**

* Valentina Giraldo Gaviria
* Héctor Hernan Betancourt López
* Marcela Fajardo Bermúdez
* Rafael Chamorro

In [ ]:
import pulp as lp
from pulp import *

#Ejercicio 1



In [ ]:

# 1. Conjuntos

nombres_cultivos = ["Maíz", "Trigo"]
nombres_fincas = ["Finca 1", "Finca 2"]

cultivos = [0, 1] # 0 = Maíz , 1 = Trigo
fincas = [0, 1]   # 0 = Finca 1, 1 = Finca 2

# 2. Parámetros

rendimiento = [[500, 650],[400,350]]
costo = [[100,120],[90,80]]
demanda = [7000,11000]
acres_disponibles = [100,100]

# 3. Modelo
modelo_finca = lp.LpProblem("Modelo_finca", sense=lp.LpMinimize)

# 4. Variables de decisión

x = lp.LpVariable.dicts("acres",(cultivos, fincas),0,cat=lp.LpContinuous)

# 5. Función objetivo

modelo_finca += lp.lpSum(costo[i][j] * x[i][j] for i in cultivos for j in fincas)

# 6. Restricciones de demanda

for i in cultivos:
    modelo_finca += lp.lpSum(rendimiento[i][j] * x[i][j] for j in fincas) >= demanda[i]

# 7. Restricciones de acres

for j in fincas:
    modelo_finca += lp.lpSum(x[i][j]for i in cultivos) <= acres_disponibles[j]

# 8. Resolver

modelo_finca.solve()

# 9. Resultados

print("=" * 60)
print("EJERCICIO 1 - Walnut Orchard")
print("=" * 60)
print("Estado:", lp.LpStatus[modelo_finca.status])
print("Costo mínimo:", lp.value(modelo_finca.objective))



for i in cultivos:
    for j in fincas:
        print(f"Acres de {nombres_cultivos[i]} en {nombres_fincas[j]}: {x[i][j].value()}")

EJERCICIO 1 - Walnut Orchard
Estado: Optimal
Costo mínimo: 3767.30772
Acres de Maíz en Finca 1: 0.0
Acres de Maíz en Finca 2: 10.769231
Acres de Trigo en Finca 1: 27.5
Acres de Trigo en Finca 2: 0.0


#Ejercicio 2

In [ ]:

# 1. Conjuntos

turnos = ["FT1", "FT2", "FT3", "PT1", "PT2"]
horas  = list(range(10, 20))   # t=10 representa 10-11 a.m., ..., t=19 representa 7-8 p.m.


# 2. Parámetros


# Matriz de cobertura: MC[t][s] = 1 si el turno s cubre la hora t, 0 si no
cobertura = {
    10: {"FT1":1, "FT2":0, "FT3":0, "PT1":0, "PT2":0},
    11: {"FT1":1, "FT2":1, "FT3":0, "PT1":0, "PT2":0},
    12: {"FT1":1, "FT2":1, "FT3":1, "PT1":0, "PT2":0},
    13: {"FT1":1, "FT2":1, "FT3":1, "PT1":0, "PT2":0},
    14: {"FT1":1, "FT2":1, "FT3":1, "PT1":1, "PT2":0},
    15: {"FT1":1, "FT2":1, "FT3":1, "PT1":1, "PT2":1},
    16: {"FT1":1, "FT2":1, "FT3":1, "PT1":1, "PT2":1},
    17: {"FT1":1, "FT2":1, "FT3":1, "PT1":1, "PT2":1},
    18: {"FT1":0, "FT2":1, "FT3":1, "PT1":1, "PT2":1},
    19: {"FT1":0, "FT2":0, "FT3":1, "PT1":0, "PT2":1},
}

# Cheques recibidos por hora (CRt)
cheques_recibidos = {
    10: 5000,
    11: 4000,
    12: 3000,
    13: 4000,
    14: 2500,
    15: 3000,
    16: 4000,
    17: 4500,
    18: 3500,
    19: 3000,
}

# Costo diario por turno (Cs)
costo = {
    "FT1": 160,
    "FT2": 160,
    "FT3": 160,
    "PT1":  75,
    "PT2":  75,
}

K = 500   # capacidad de procesamiento por máquina/hora
M = 13    # máquinas disponibles


# 3. Crear modelo


modelo = lp.LpProblem("Bank_One", lp.LpMinimize)


# 4. Variables de decisión


# Xs: número de trabajadores asignados al turno s (entero no negativo)
x = lp.LpVariable.dicts(
    "X",
    turnos,
    lowBound=0,
    cat="Integer"
)

# It: inventario de cheques acumulados al final de la hora t (continua no negativa)
I = lp.LpVariable.dicts(
    "I",
    horas,
    lowBound=0,
    cat="Continuous"
)


# 5. Función objetivo


# Minimizar: sum_s (Cs * Xs)
modelo += lp.lpSum(
    costo[s] * x[s]
    for s in turnos
)



# 6. Restricciones de inventario (cobertura de cheques)


# It = I(t-1) + CRt - K * sum_s(MC(t,s) * Xs)   para todo t
# con I9 = 0 (no hay inventario antes de abrir)

for t in horas:
    trabajadores_activos = lp.lpSum(cobertura[t][s] * x[s] for s in turnos)
    inventario_anterior  = 0 if t == 10 else I[t - 1]
    modelo += (
        I[t] == inventario_anterior + cheques_recibidos[t] - K * trabajadores_activos
    ), f"Inventario_t{t}"


# -----------------------------
# 7. Restricción de meta: todos los cheques procesados a las 8 p.m.
# -----------------------------

modelo += I[19] == 0, "Meta_final"


# -----------------------------
# 8. Restricciones de capacidad de máquinas
# -----------------------------

# Los trabajadores activos en cada hora no pueden superar las 13 máquinas
for t in horas:
    modelo += (
        lp.lpSum(cobertura[t][s] * x[s] for s in turnos) <= M
    ), f"Capacidad_t{t}"


# -----------------------------
# 9. Restricción de mínimo de trabajadores full-time
# -----------------------------

modelo += (
    lp.lpSum(x[s] for s in ["FT1", "FT2", "FT3"]) >= 3
), "MinFT"


# -----------------------------
# 10. Resolver
# -----------------------------

modelo.solve()


# -----------------------------
# 11. Resultados
# -----------------------------
print("=" * 60)
print("EJERCICIO 2 - Bank One")
print("=" * 60)

print("Estado:", lp.LpStatus[modelo.status])
print(f"Costo mínimo: ${lp.value(modelo.objective):,.0f}/día")
print()
print("--- Trabajadores por turno ---")
for s in turnos:
    print(f"  X{s} = {int(x[s].value())} trabajadores")
print()
print("--- Inventario de cheques al final de cada hora ---")
horas_label = {
    10:"10 a.m.",11:"11 a.m.",12:"Noon",13:"1 p.m.",14:"2 p.m.",
    15:"3 p.m.",16:"4 p.m.",17:"5 p.m.",18:"6 p.m.",19:"7 p.m."
}
for t in horas:
    print(f"  I({horas_label[t]}) = {I[t].value():,.0f} cheques pendientes")

EJERCICIO 2 - Bank One
Estado: Optimal
Costo mínimo: $1,335/día

--- Trabajadores por turno ---
  XFT1 = 0 trabajadores
  XFT2 = 5 trabajadores
  XFT3 = 1 trabajadores
  XPT1 = 0 trabajadores
  XPT2 = 5 trabajadores

--- Inventario de cheques al final de cada hora ---
  I(10 a.m.) = 5,000 cheques pendientes
  I(11 a.m.) = 6,500 cheques pendientes
  I(Noon) = 6,500 cheques pendientes
  I(1 p.m.) = 7,500 cheques pendientes
  I(2 p.m.) = 7,000 cheques pendientes
  I(3 p.m.) = 4,500 cheques pendientes
  I(4 p.m.) = 3,000 cheques pendientes
  I(5 p.m.) = 2,000 cheques pendientes
  I(6 p.m.) = 0 cheques pendientes
  I(7 p.m.) = 0 cheques pendientes


#Ejercicio 3

**Sunco Oil** — Maximizar utilidad neta (operacion 10 anos menos expansion).

**Modelo (PL):**

MAX Z = 10*(20000*X_LH + 15000*X_LN + 18000*X_CH + 17000*X_CN) - 120000*E_L - 150000*E_C

In [ ]:
# 1. Conjuntos

nombres_refinerias = ["Los Angeles", "Chicago"]
nombres_destinos   = ["Houston", "Nueva York"]

refinerias = [0, 1]  # 0 = Los Angeles, 1 = Chicago
destinos   = [0, 1]  # 0 = Houston,     1 = Nueva York

# 2. Parámetros

# Gij: ganancia por millon de barriles de refineria i a destino j ($/M bbl)
#              Houston  Nueva York
ganancia = [[20000,    15000],   # Los Angeles (i=0)
            [18000,    17000]]   # Chicago     (i=1)

# Ci: capacidad actual de cada refineria (millones bbl/ano)
capacidad = [2, 3]  # C1=2 (LA), C2=3 (Chicago)

# Dj: demanda maxima de cada destino (millones bbl/ano)
demanda = [5, 5]    # D1=5 (Houston), D2=5 (Nueva York)

# Ei: costo de expansion por millon de barriles adicional ($/M bbl)
costo_expansion = [120000, 150000]  # E1=120000 (LA), E2=150000 (Chicago)

# T: horizonte de tiempo (anos)
T = 10

# 3. Modelo

modelo_sunco = lp.LpProblem("Sunco_Oil", sense=lp.LpMaximize)

# 4. Variables de decision

# Xij: millones de barriles enviados de refineria i a destino j (por ano)
x = lp.LpVariable.dicts("X", (refinerias, destinos), 0, cat=lp.LpContinuous)

# Yi: millones de barriles de capacidad extra en refineria i (una sola vez)
y = lp.LpVariable.dicts("Y", refinerias, 0, cat=lp.LpContinuous)

# 5. Funcion objetivo
# MAX Z = T * SUM(Gij * Xij) - SUM(Ei * Yi)
# Ganancias anuales x 10 anos MENOS costos de expansion (pago unico)

modelo_sunco += (T * lp.lpSum(ganancia[i][j] * x[i][j]
                               for i in refinerias
                               for j in destinos)
                 - lp.lpSum(costo_expansion[i] * y[i]
                             for i in refinerias)), "Utilidad_Neta_10_anos"

# 6. Restricciones de capacidad de refinacion
# Lo que sale de cada refineria <= capacidad actual + expansion
# SUM_j(Xij) <= Ci + Yi  para todo i

for i in refinerias:
    modelo_sunco += (lp.lpSum(x[i][j] for j in destinos)
                     <= capacidad[i] + y[i],
                     f"Cap_{nombres_refinerias[i]}")

# 7. Restricciones de demanda maxima
# Lo que llega a cada destino <= demanda maxima
# SUM_i(Xij) <= Dj  para todo j

for j in destinos:
    modelo_sunco += (lp.lpSum(x[i][j] for i in refinerias)
                     <= demanda[j],
                     f"Dem_{nombres_destinos[j]}")

# 8. Resolver

modelo_sunco.solve(PULP_CBC_CMD(msg=0))

# 9. Resultados

print("=" * 60)
print("EJERCICIO 3 - Sunco Oil")
print("=" * 60)
print(f"Estado                : {lp.LpStatus[modelo_sunco.status]}")
print(f"Utilidad neta 10 anos : ${lp.value(modelo_sunco.objective):,.0f}")
print()
print("Flujo optimo de envios (millones bbl/ano):")
for i in refinerias:
    for j in destinos:
        print(f"  {nombres_refinerias[i]:12} -> {nombres_destinos[j]:10}: "
              f"{x[i][j].value():.2f} M bbl/ano")
print()
print("Expansion de refinerias:")
for i in refinerias:
    print(f"  Expansion {nombres_refinerias[i]:12}: "
          f"{y[i].value():.2f} M bbl/ano adicionales")
print()
print("Uso de capacidad por refineria:")
for i in refinerias:
    uso = sum(x[i][j].value() for j in destinos)
    cap_total = capacidad[i] + y[i].value()
    print(f"  {nombres_refinerias[i]:12}: {uso:.2f} / {cap_total:.2f} M bbl/ano")

EJERCICIO 3 - Sunco Oil
Estado                : Optimal
Utilidad neta 10 anos : $1,210,000

Flujo optimo de envios (millones bbl/ano):
  Los Angeles  -> Houston   : 5.00 M bbl/ano
  Los Angeles  -> Nueva York: 2.00 M bbl/ano
  Chicago      -> Houston   : 0.00 M bbl/ano
  Chicago      -> Nueva York: 3.00 M bbl/ano

Expansion de refinerias:
  Expansion Los Angeles : 5.00 M bbl/ano adicionales
  Expansion Chicago     : 0.00 M bbl/ano adicionales

Uso de capacidad por refineria:
  Los Angeles : 7.00 / 7.00 M bbl/ano
  Chicago     : 3.00 / 3.00 M bbl/ano


# Ejercicio 4

In [ ]:

# 1. Conjuntos

diagnostico = range(4)

# 2. Parámetros

ganancia = [2000, 1500,500,300]
servicio= [7,4,2,1]
cama = [5,2,1,0]
enfermeria = [30,10,5,1]
medicamento= [800,500,150,50]
demanda=[10,15,40,160]

# 3. Modelo
modelo_drg = lp.LpProblem("Modelo_DRG", sense=lp.LpMaximize)

# 4. Variables de decisión

x = lp.LpVariable.dicts("casos",(diagnostico),0,cat='Integer')

# 5. Función objetivo

modelo_drg += lp.lpSum(ganancia[i] * x[i] for i in diagnostico)

# 6. Restricciones de diagnostico

modelo_drg += lp.lpSum(servicio[i] * x[i] for i in diagnostico) <= 570

# 7. Restricciones de cama

modelo_drg += lp.lpSum(cama[i] * x[i] for i in diagnostico) <= 1000

# 8. Restricciones de enfermeria

modelo_drg += lp.lpSum(enfermeria[i] * x[i] for i in diagnostico) <= 50000

# 9. Restricciones de medicamento

modelo_drg += lp.lpSum(medicamento[i] * x[i] for i in diagnostico) <= 50000

# 9. Restricciones de demanda

for i in diagnostico:
    modelo_drg += x[i] >= demanda[i]

# 10. Resolver

modelo_drg.solve()

# 11. Resultados

print("=" * 60)
print("EJERCICIO 4 - Gotham City Hospital")
print("=" * 60)

print("Estado:", lp.LpStatus[modelo_drg.status])
print("Costo mínimo:", lp.value(modelo_drg.objective))


for i in diagnostico:
      print(f"Casos del grupo DRG {i+1}: {x[i].value()}")

EJERCICIO 4 - Gotham City Hospital
Estado: Optimal
Costo mínimo: 181000.0
Casos del grupo DRG 1: 10.0
Casos del grupo DRG 2: 50.0
Casos del grupo DRG 3: 40.0
Casos del grupo DRG 4: 220.0


#Ejercicio 5

In [ ]:


# ============================================================
# 1. Conjuntos
# ============================================================

nombres_tipos = ["Sencillas", "Dobles", "Triples", "Cuádruples"]

tipos = [0, 1, 2, 3]
# 0 = Sencillas
# 1 = Dobles
# 2 = Triples
# 3 = Cuádruples


# ============================================================
# 2. Parámetros
# ============================================================

# Impuesto generado por cada tipo de casa
impuesto = [1000, 1900, 2700, 3400]

# Costo de construcción por unidad
costo_construccion = [50000, 70000, 130000, 160000]

# Área requerida por cada tipo de casa, en acres
area = [0.18, 0.28, 0.40, 0.50]

# Datos de demolición
max_casas_demolidas = 300
area_por_casa_demolida = 0.25
porcentaje_area_util = 0.85
costo_demolicion = 2000

# Área útil que se obtiene por cada casa popular demolida
area_util_por_demolicion = area_por_casa_demolida * porcentaje_area_util

# Presupuesto máximo disponible del Banco
presupuesto = 15000000


# ============================================================
# 3. Modelo
# ============================================================

modelo_erstville = lp.LpProblem("Modelo_Erstville", sense=lp.LpMaximize)


# ============================================================
# 4. Variables de decisión
# ============================================================

# x[t] representa el número de casas del tipo t que se construirán
x = lp.LpVariable.dicts("viviendas", tipos, lowBound=0, cat=lp.LpInteger)

# y representa el número de casas populares demolidas
y = lp.LpVariable("casas_demolidas", lowBound=0, cat=lp.LpInteger)


# ============================================================
# 5. Función objetivo
# ============================================================

# Maximizar la recaudación total de impuestos
modelo_erstville += lp.lpSum(impuesto[t] * x[t] for t in tipos)


# ============================================================
# 6. Restricción de presupuesto
# ============================================================

# El costo de construcción más el costo de demolición
# no puede superar los 15 millones de dólares

modelo_erstville += (
    lp.lpSum(costo_construccion[t] * x[t] for t in tipos)
    + costo_demolicion * y
    <= presupuesto
)


# ============================================================
# 7. Restricción de área disponible
# ============================================================

# El área ocupada por las casas nuevas no puede superar
# el área útil obtenida por las casas populares demolidas

modelo_erstville += (
    lp.lpSum(area[t] * x[t] for t in tipos)
    <= area_util_por_demolicion * y
)


# ============================================================
# 8. Restricción de máximo número de casas demolidas
# ============================================================

modelo_erstville += y <= max_casas_demolidas


# ============================================================
# 9. Restricción de triples y cuádruples
# ============================================================

# Las casas triples y cuádruples deben ocupar al menos
# el 25% del área total construida

modelo_erstville += (
    area[2] * x[2] + area[3] * x[3]
    >= 0.25 * lp.lpSum(area[t] * x[t] for t in tipos)
)


# ============================================================
# 10. Restricción de viviendas sencillas
# ============================================================

# Las casass sencillas deben representar al menos
# el 20% del total de viviendas construidas

modelo_erstville += (
    x[0] >= 0.20 * lp.lpSum(x[t] for t in tipos)
)


# ============================================================
# 11. Restricción de viviendas dobles
# ============================================================

# Las casas dobles deben ocupar al menos
# el 10% del área total construida

modelo_erstville += (
    area[1] * x[1]
    >= 0.10 * lp.lpSum(area[t] * x[t] for t in tipos)
)


# ============================================================
# 12. Resolver
# ============================================================

modelo_erstville.solve()


# ============================================================
# 13. Resultados
# ============================================================

print("=" * 60)
print("EJERCICIO 5 - Ciudad de Erstville")
print("=" * 60)

print("Estado:", lp.LpStatus[modelo_erstville.status])
print("Recaudación máxima de impuestos:", lp.value(modelo_erstville.objective))

print("\nPlan óptimo de construcción:")
for t in tipos:
    print(f"Casas {nombres_tipos[t]}: {x[t].value()}")

print(f"Casas demolidas: {y.value()}")


# ============================================================
# 14. Verificación de restricciones
# ============================================================

total_viviendas = sum(x[t].value() for t in tipos)
area_total = sum(area[t] * x[t].value() for t in tipos)
area_disponible = area_util_por_demolicion * y.value()

costo_total = (
    sum(costo_construccion[t] * x[t].value() for t in tipos)
    + costo_demolicion * y.value()
)

area_triples_cuadruples = area[2] * x[2].value() + area[3] * x[3].value()
area_dobles = area[1] * x[1].value()

print("\nVerificación del modelo:")
print("Costo total del proyecto:", costo_total)
print("Presupuesto disponible:", presupuesto)

print("Área total construida:", area_total)
print("Área disponible para construir:", area_disponible)

print("Total de viviendas construidas:", total_viviendas)

print("Porcentaje de viviendas sencillas:", x[0].value() / total_viviendas)
print("Porcentaje mínimo requerido de sencillas: 0.20")

print("Porcentaje de área en triples y cuádruples:", area_triples_cuadruples / area_total)
print("Porcentaje mínimo requerido: 0.25")

print("Porcentaje de área en dobles:", area_dobles / area_total)
print("Porcentaje mínimo requerido: 0.10")

EJERCICIO 5 - Ciudad de Erstville
Estado: Optimal
Recaudación máxima de impuestos: 354200.0

Plan óptimo de construcción:
Casas Sencillas: 37.0
Casas Dobles: 119.0
Casas Triples: 1.0
Casas Cuádruples: 26.0
Casas demolidas: 252.0

Verificación del modelo:
Costo total del proyecto: 14974000.0
Presupuesto disponible: 15000000
Área total construida: 53.38
Área disponible para construir: 53.55
Total de viviendas construidas: 183.0
Porcentaje de viviendas sencillas: 0.20218579234972678
Porcentaje mínimo requerido de sencillas: 0.20
Porcentaje de área en triples y cuádruples: 0.2510303484451105
Porcentaje mínimo requerido: 0.25
Porcentaje de área en dobles: 0.6242038216560509
Porcentaje mínimo requerido: 0.10
